# QML-SleepNet — FINAL Fixed Fusion Post-hoc x Scorer v1

**Run only if the development audit printed `PROMOTE_FUSION: True`.**

This notebook verifies the frozen fusion SHA/manifest first, then opens the official x labels solely for one-time scoring.

No training, weight search, calibration, HMM fitting, threshold tuning, or method change is possible here.

Because the parent x results were historically known before this fusion audit was conceived, any fused x result is **post-hoc historical-benchmark evidence**, not pristine external validation.

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, uuid

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    confusion_matrix, brier_score_loss, log_loss
)

ROOT = Path("/content/drive/MyDrive/QML_SleepNet")
OUT = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX"
    / "FINAL_METRIC_CHAMPION_FIXED_FUSION_AUDIT_v1"
)
FROZEN = OUT / "FINAL_FIXED_FUSION_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"
MANIFEST = OUT / "FINAL_FIXED_FUSION_PRE_SCORE_FREEZE_MANIFEST.json"

RAW = ROOT / "data/raw/apnea_ecg"
LABEL_JSON = RAW / "test_set_apnea_labels.json"
LABEL_TXT = RAW / "test-dataset-annos.txt"

for p in [FROZEN, MANIFEST, LABEL_JSON, LABEL_TXT]:
    if not p.is_file():
        raise FileNotFoundError(p)


Mounted at /content/drive


In [2]:
def sha256_file(path, chunk=1<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def atomic_json(path,obj):
    path=Path(path)
    tmp=path.with_name(path.name+f".tmp.{uuid.uuid4().hex}")
    tmp.write_text(json.dumps(obj,indent=2,sort_keys=True,default=str))
    os.replace(tmp,path)

def metric_row(y,score,threshold=0.5):
    y=np.asarray(y,dtype=np.int8); score=np.asarray(score,dtype=float)
    pred=(score>=threshold).astype(np.int8)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {
        "n":int(len(y)),
        "accuracy":float(accuracy_score(y,pred)),
        "balanced_accuracy":float(balanced_accuracy_score(y,pred)),
        "precision":float(precision_score(y,pred,zero_division=0)),
        "sensitivity":float(recall_score(y,pred,zero_division=0)),
        "specificity":float(tn/max(tn+fp,1)),
        "f1":float(f1_score(y,pred,zero_division=0)),
        "mcc":float(matthews_corrcoef(y,pred)),
        "auroc":float(roc_auc_score(y,score)),
        "auprc":float(average_precision_score(y,score)),
        "brier":float(brier_score_loss(y,score)),
        "nll":float(log_loss(y,np.column_stack([1-score,score]),labels=[0,1])),
        "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
    }

def normalize_json_labels(obj):
    out={}
    for rec,val in obj.items():
        seq=val.strip() if isinstance(val,str) else "".join(map(str,val))
        if set(seq)-{"A","N"}:
            raise RuntimeError(f"{rec}: unexpected label symbol")
        out[str(rec)]=seq
    return out

def parse_txt(path):
    out={}; current=None; chunks=[]
    for raw in Path(path).read_text().splitlines():
        line=raw.strip()
        if not line: continue
        if len(line)==3 and line.startswith("x") and line[1:].isdigit():
            if current is not None: out[current]="".join(chunks)
            current=line; chunks=[]; continue
        if current is None: continue
        parts=line.split()
        if len(parts)>=2 and parts[0].isdigit():
            seq="".join(parts[1:]).strip()
            if set(seq)-{"A","N"}:
                raise RuntimeError(line)
            chunks.append(seq)
    if current is not None:
        out[current]="".join(chunks)
    return out


In [3]:
# Verify freeze before labels.
manifest=json.loads(MANIFEST.read_text())
if manifest.get("official_x_labels_loaded") is not False:
    raise RuntimeError("Fusion development notebook provenance failure")
if manifest.get("training_performed") is not False:
    raise RuntimeError("Unexpected training in fusion audit")

got=sha256_file(FROZEN)
if got != manifest["fused_prediction_sha256"]:
    raise RuntimeError("Frozen fusion SHA mismatch")

z=np.load(FROZEN,allow_pickle=False)
UID=np.asarray(z["test_uids"]).astype(str)
SCORE=np.asarray(z["fused_probability"],dtype=float)
PRED=np.asarray(z["prediction"],dtype=np.int8)
THR=float(np.asarray(z["hard_threshold"]).item())

if len(UID)!=17248:
    raise RuntimeError(f"Expected 17,248 rows, got {len(UID)}")
if not np.isfinite(SCORE).all():
    raise RuntimeError("Non-finite frozen score")
if not np.array_equal(PRED,(SCORE>=THR).astype(np.int8)):
    raise RuntimeError("Frozen prediction/score mismatch")

print("Freeze verified:",got)
print("Winner:",manifest["winning_fusion"]["candidate"])
print("Weights:",
      manifest["winning_fusion"]["w_phys"],
      manifest["winning_fusion"]["w_qml"])
print("NOW official labels may be opened solely for scoring.")


Freeze verified: 063a017e61188393bcdcdacb72958ffa3e7e0efa9432d33aa0845983462dfa1f
Winner: FIXED_LOGIT_PHYS_25_QML_75
Weights: 0.25 0.75
NOW official labels may be opened solely for scoring.


In [4]:
# Open and cross-check the official x labels.
j=normalize_json_labels(json.loads(LABEL_JSON.read_text()))
t=parse_txt(LABEL_TXT)
records=[f"x{i:02d}" for i in range(1,36)]

if sorted(j)!=records or sorted(t)!=records:
    raise RuntimeError("Official x record universe mismatch")
for r in records:
    if j[r]!=t[r]:
        raise RuntimeError(f"Official label sources disagree for {r}")

# UID format is record:epoch.
Y=np.full(len(UID),-1,dtype=np.int8)
bad=[]
for i,u in enumerate(UID):
    r,e=u.rsplit(":",1)
    e=int(e)
    if r not in j or e<0 or e>=len(j[r]):
        bad.append(u); continue
    Y[i]=1 if j[r][e]=="A" else 0

if bad or np.any(Y<0):
    raise RuntimeError(f"Official-label alignment failure: {bad[:10]}")

print("Official JSON/TXT agreement: PASS")
print("Rows aligned:",len(Y))


Official JSON/TXT agreement: PASS
Rows aligned: 17248


In [5]:
FINAL=metric_row(Y,SCORE,THR)

CAT_REFERENCE={
    "name":"Stage15A physiology CatBoost + temperature + HMM",
    "accuracy":0.9101924860859224,
    "balanced_accuracy":0.9097461483185166,
    "precision":0.8627049180327869,
    "sensitivity":0.9079016603679574,
    "specificity":0.9115906369863014,
    "f1":0.884726368159204,
    "mcc":0.811955,
    "auroc":0.96966,
    "auprc":0.95045,
}
QML_REFERENCE={
    "name":"Guide-primary Bridge + QT + corrected Stage06 fixed ensemble",
    "accuracy":0.9011479591836735,
    "balanced_accuracy":0.9001158045109892,
    "precision":0.8514808362369338,
    "sensitivity":0.8958301512142967,
    "specificity":0.9044014578076816,
    "f1":0.8730926684034239,
    "mcc":0.7929076491751258,
    "auroc":0.9620666432037354,
    "auprc":0.9427717592257048,
}

keys=["accuracy","balanced_accuracy","precision","sensitivity","specificity",
      "f1","mcc","auroc","auprc"]

report={
    "scientific_status":"post-hoc historical-benchmark fused evidence",
    "frozen_fusion_sha256":got,
    "fusion":FINAL,
    "stage15a_reference":CAT_REFERENCE,
    "qml_reference":QML_REFERENCE,
    "fusion_minus_stage15a":{
        k:float(FINAL[k]-CAT_REFERENCE[k]) for k in keys
    },
    "fusion_minus_qml":{
        k:float(FINAL[k]-QML_REFERENCE[k]) for k in keys
    },
    "beats_91p0192_accuracy":bool(FINAL["accuracy"]>CAT_REFERENCE["accuracy"]),
    "method_changed_after_label_open":False,
}
atomic_json(OUT/"FINAL_FIXED_FUSION_POSTHOC_X_METRICS.json",report)

print("="*110)
print("FINAL FIXED FUSION — POST-HOC X RESULT")
print("="*110)
print(json.dumps(report,indent=2))


FINAL FIXED FUSION — POST-HOC X RESULT
{
  "scientific_status": "post-hoc historical-benchmark fused evidence",
  "frozen_fusion_sha256": "063a017e61188393bcdcdacb72958ffa3e7e0efa9432d33aa0845983462dfa1f",
  "fusion": {
    "n": 17248,
    "accuracy": 0.9086270871985158,
    "balanced_accuracy": 0.9076848836674374,
    "precision": 0.8621594055077955,
    "sensitivity": 0.9037727203299221,
    "specificity": 0.9115970470049528,
    "f1": 0.882475764354959,
    "mcc": 0.8084026100817799,
    "auroc": 0.9675236374617686,
    "auprc": 0.9497755803318464,
    "brier": 0.0796448443135515,
    "nll": 0.4334665417947636,
    "tn": 9755,
    "fp": 946,
    "fn": 630,
    "tp": 5917
  },
  "stage15a_reference": {
    "name": "Stage15A physiology CatBoost + temperature + HMM",
    "accuracy": 0.9101924860859224,
    "balanced_accuracy": 0.9097461483185166,
    "precision": 0.8627049180327869,
    "sensitivity": 0.9079016603679574,
    "specificity": 0.9115906369863014,
    "f1": 0.88472636815920